# Video → Frames → Perceptual pre-dedup (Stages 1–2 of the pipeline)

This notebook covers the first three components of the content-verification pipeline:

1. **Video upload** — direct upload, Google Drive, or an existing path
2. **Frame extraction agent** — ffmpeg samples 1 frame every 2 minutes (robust to video streams that end before the container does), with a timestamped `manifest.json`
3. **Perceptual pre-dedup agent** — pHash + dHash moves visually near-identical frames aside *before* any GPU cost

Output: `frames_kept/` + `manifest.json`, zipped — exactly what the Qwen content-extraction stage consumes.

**Run the cells top to bottom.** Each agent is idempotent — you can re-run it safely.

In [ ]:
# @title 1. Setup — install dependencies { display-mode: "form" }
# ffmpeg ships with Colab, so only Python deps are needed.
!pip install -q imagehash pillow

import json, math, shutil, subprocess, sys, time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from datetime import datetime, timezone

import imagehash
from PIL import Image

# Verify ffmpeg / ffprobe are actually present
for tool in ("ffmpeg", "ffprobe"):
    r = subprocess.run([tool, "-version"], capture_output=True, text=True)
    assert r.returncode == 0, f"{tool} not found — run: !apt-get install -y ffmpeg"
    print(r.stdout.splitlines()[0])
print("\n✅ Environment ready")


In [ ]:
# @title 2. Pipeline configuration { display-mode: "form" }

@dataclass
class PipelineConfig:
    # --- frame extraction ---
    frame_interval_sec: int = 120          # 1 frame every 2 minutes
    image_format: str = "jpg"              # jpg keeps size down; use "png" for lossless
    jpeg_quality: int = 2                  # ffmpeg -q:v scale: 2 = high quality, 31 = worst
    accurate_seek: bool = False            # True = one ffmpeg seek per frame (exact timestamps, slower)
                                           # False = single-pass fps filter (fast, timestamps ≈ n*interval)
    scale_max_width: int = 1280            # downscale huge videos; 1280px is plenty for slide OCR.
                                           # set to 0 to keep original resolution

    # --- perceptual dedup ---
    hash_size: int = 16                    # 16 → 256-bit hashes (finer than default 8)
    phash_threshold: int = 12              # Hamming distance below which frames count as duplicates
    dhash_threshold: int = 12              #   (out of hash_size^2 = 256 bits; ~5% of bits)
    require_both: bool = True              # True: BOTH pHash and dHash must agree it's a dupe
                                           #   (conservative — fewer false deletions)
    compare_window: int = 3                # compare against the last N *kept* frames, not just 1

    # --- paths ---
    work_dir: str = "/content/pipeline"

    def __post_init__(self):
        self.frames_dir     = str(Path(self.work_dir) / "frames_raw")
        self.kept_dir       = str(Path(self.work_dir) / "frames_kept")
        self.duplicates_dir = str(Path(self.work_dir) / "frames_duplicates")
        self.manifest_path  = str(Path(self.work_dir) / "manifest.json")

CFG = PipelineConfig()

for d in (CFG.work_dir, CFG.frames_dir, CFG.kept_dir, CFG.duplicates_dir):
    Path(d).mkdir(parents=True, exist_ok=True)

print(json.dumps({k: v for k, v in asdict(CFG).items()}, indent=2))


## Stage 0 — Video upload

Pick **one** of the three options below. For anything over ~200 MB, Google Drive (Option B) is far more reliable than the browser uploader.

> ⚠️ Make sure you have legitimate offline access to the video (e.g. Pluralsight's own offline feature on your account). Don't feed this pipeline content you aren't licensed to store.

In [ ]:
# @title Option A — direct browser upload (good for small files < ~200 MB)
from google.colab import files

uploaded = files.upload()  # opens a file picker
VIDEO_PATH = str(Path("/content") / next(iter(uploaded.keys())))
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title Option B — mount Google Drive (recommended for large files)
from google.colab import drive
drive.mount("/content/drive")

# ── EDIT THIS to point at your file inside Drive ──
VIDEO_PATH = "/content/drive/MyDrive/videos/course.mp4"

assert Path(VIDEO_PATH).exists(), f"Not found: {VIDEO_PATH} — fix the path above"
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title Option C — file already on the Colab disk (e.g. wget/rsync'd earlier)
VIDEO_PATH = "/content/course.mp4"   # ← edit
assert Path(VIDEO_PATH).exists(), f"Not found: {VIDEO_PATH}"
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title 3. Probe the video (duration, resolution, fps) and sanity-check

def probe_video(path: str) -> dict:
    """Return container + first-video-stream metadata via ffprobe."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,codec_name",
        "-show_entries", "format=duration,size,format_name",
        "-of", "json", path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffprobe failed:\n{r.stderr}")
    meta = json.loads(r.stdout)
    stream, fmt = meta["streams"][0], meta["format"]
    num, den = (stream.get("avg_frame_rate") or "0/1").split("/")
    fps = (float(num) / float(den)) if float(den) else 0.0
    return {
        "path": path,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "fps": round(fps, 3),
        "duration_sec": float(fmt["duration"]),
        "size_mb": round(int(fmt["size"]) / 1e6, 1),
        "container": fmt.get("format_name"),
    }

VIDEO_INFO = probe_video(VIDEO_PATH)
expected_frames = math.floor(VIDEO_INFO["duration_sec"] / CFG.frame_interval_sec) + 1

print(json.dumps(VIDEO_INFO, indent=2))
print(f"\nDuration: {VIDEO_INFO['duration_sec']/60:.1f} min "
      f"→ expecting ~{expected_frames} frames at 1 per {CFG.frame_interval_sec}s")

if VIDEO_INFO["duration_sec"] < CFG.frame_interval_sec:
    print("⚠️ Video is shorter than the sampling interval — you'll get a single frame.")


## Stage 1 — Frame extraction agent

Two extraction modes, controlled by `CFG.accurate_seek`:

- **Fast (default)** — one ffmpeg pass with `fps=1/120`. Frame *n* lands at *t ≈ (n−1)·120s*. For lecture content this is exact enough, and it's a single decode of the file.
- **Accurate** — one `ffmpeg -ss <t>` seek per timestamp. Frame-exact, but launches a process per frame (still fast because `-ss` before `-i` uses keyframe seeking).

Every extracted frame gets a manifest entry carrying its timestamp forward — the final coverage report depends on these.

In [ ]:
# @title 4. Frame extraction agent

class FrameExtractionAgent:
    """Extracts 1 frame every `frame_interval_sec` seconds and builds a timestamped manifest."""

    def __init__(self, cfg: PipelineConfig, video_info: dict):
        self.cfg, self.info = cfg, video_info

    # ---------- public ----------
    def run(self) -> dict:
        out_dir = Path(self.cfg.frames_dir)
        for old in out_dir.glob(f"*.{self.cfg.image_format}"):
            old.unlink()                                # idempotent re-runs

        t0 = time.time()
        if self.cfg.accurate_seek:
            frames = self._extract_accurate(out_dir)
        else:
            frames = self._extract_fast(out_dir)
        elapsed = time.time() - t0

        manifest = {
            "video": self.info,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "config": asdict(self.cfg),
            "stages": {"extraction": {"mode": "accurate" if self.cfg.accurate_seek else "fast",
                                      "elapsed_sec": round(elapsed, 1),
                                      "frame_count": len(frames)}},
            "frames": frames,
        }
        Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"✅ Extracted {len(frames)} frames in {elapsed:.1f}s → {out_dir}")
        return manifest

    # ---------- internals ----------
    def _scale_filter(self) -> str:
        w = self.cfg.scale_max_width
        return f",scale='min({w},iw)':-2" if w else ""

    def _frame_record(self, path: Path, index: int, ts: float) -> dict:
        return {
            "frame_id": path.stem,
            "index": index,
            "timestamp_sec": round(ts, 2),
            "timestamp_hms": time.strftime("%H:%M:%S", time.gmtime(ts)),
            "path": str(path),
            "status": "extracted",        # → kept / duplicate after dedup
            "phash": None, "dhash": None,
            "dedup": None,
        }

    def _extract_fast(self, out_dir: Path) -> list:
        pattern = str(out_dir / f"frame_%04d.{self.cfg.image_format}")
        dur = self.info["duration_sec"]
        # tpad clones the last real frame out to the container duration: screen recordings
        # often stop emitting video frames before the audio/container ends, which would
        # otherwise silently truncate sampling (e.g. 7 frames from a 14.5-min video).
        vf = (f"tpad=stop_mode=clone:stop_duration={dur},"
              f"fps=1/{self.cfg.frame_interval_sec}{self._scale_filter()}")
        cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
               "-i", self.info["path"], "-vf", vf,
               "-t", f"{dur + self.cfg.frame_interval_sec / 2:.3f}",
               "-q:v", str(self.cfg.jpeg_quality),
               "-vsync", "vfr",   # NOT -fps_mode: Colab ships ffmpeg 4.x, which lacks it
               pattern]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg failed:\n{r.stderr}")
        frames = []
        for i, p in enumerate(sorted(out_dir.glob(f"frame_*.{self.cfg.image_format}"))):
            frames.append(self._frame_record(p, i, i * self.cfg.frame_interval_sec))
        return frames

    def _extract_accurate(self, out_dir: Path) -> list:
        frames, ts, i = [], 0.0, 0
        while ts < self.info["duration_sec"]:
            p = out_dir / f"frame_{i:04d}.{self.cfg.image_format}"
            vf = f"scale='min({self.cfg.scale_max_width},iw)':-2" if self.cfg.scale_max_width else "null"
            cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                   "-ss", f"{ts:.3f}", "-i", self.info["path"],
                   "-frames:v", "1", "-vf", vf,
                   "-q:v", str(self.cfg.jpeg_quality), str(p)]
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg failed at t={ts}:\n{r.stderr}")
            if not p.exists() and frames:
                # timestamp is past the last real video frame — clone the previous one
                shutil.copy(frames[-1]["path"], p)
            if p.exists():
                frames.append(self._frame_record(p, i, ts))
            ts += self.cfg.frame_interval_sec
            i += 1
            if i % 10 == 0:
                print(f"  … {i} frames ({ts/60:.0f} min in)")
        return frames


extractor = FrameExtractionAgent(CFG, VIDEO_INFO)
manifest = extractor.run()

expected = math.floor(VIDEO_INFO["duration_sec"] / CFG.frame_interval_sec) + 1
got = len(manifest["frames"])
if got < expected:
    print(f"⚠️ Expected ~{expected} frames for a "
          f"{VIDEO_INFO['duration_sec']/60:.1f}-min video but got {got}. "
          f"The video stream may end before the container does.")
else:
    print(f"Frame count matches expectation ({got}/{expected}).")

for f in manifest["frames"]:      # full listing
    print(f'{f["frame_id"]}  @ {f["timestamp_hms"]}')


## Stage 2 — Perceptual pre-dedup agent

Two complementary hashes per frame:

- **pHash** (DCT-based) — robust to compression noise and small global changes
- **dHash** (gradient-based) — sensitive to layout shifts, cheap

A frame is a duplicate when its Hamming distance to **any of the last N kept frames** is under threshold (`require_both=True` demands both hashes agree — conservative, so you never lose a genuinely new slide). Duplicates are **moved, not deleted**, to `frames_duplicates/` so you can audit before committing.

In [ ]:
# @title 5. Perceptual pre-dedup agent

class PerceptualDedupAgent:
    """Drops visually near-identical frames using pHash + dHash Hamming distance
    against a sliding window of recently *kept* frames."""

    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg

    def run(self, manifest: dict) -> dict:
        cfg = self.cfg
        kept_window = []          # [(phash, dhash, frame_id), …] most recent last
        n_kept = n_dup = 0
        t0 = time.time()

        for rec in manifest["frames"]:
            img = Image.open(rec["path"])
            ph = imagehash.phash(img, hash_size=cfg.hash_size)
            dh = imagehash.dhash(img, hash_size=cfg.hash_size)
            rec["phash"], rec["dhash"] = str(ph), str(dh)

            match = self._find_match(ph, dh, kept_window)
            if match:
                rec["status"] = "duplicate"
                rec["dedup"] = match
                dst = Path(cfg.duplicates_dir) / Path(rec["path"]).name
                shutil.move(rec["path"], dst)
                rec["path"] = str(dst)
                n_dup += 1
            else:
                rec["status"] = "kept"
                dst = Path(cfg.kept_dir) / Path(rec["path"]).name
                shutil.move(rec["path"], dst)
                rec["path"] = str(dst)
                kept_window.append((ph, dh, rec["frame_id"]))
                kept_window = kept_window[-cfg.compare_window:]
                n_kept += 1

        manifest["stages"]["perceptual_dedup"] = {
            "kept": n_kept, "duplicates": n_dup,
            "reduction_pct": round(100 * n_dup / max(1, n_kept + n_dup), 1),
            "phash_threshold": cfg.phash_threshold,
            "dhash_threshold": cfg.dhash_threshold,
            "require_both": cfg.require_both,
            "elapsed_sec": round(time.time() - t0, 1),
        }
        Path(cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"✅ Kept {n_kept}, moved {n_dup} duplicates "
              f"({manifest['stages']['perceptual_dedup']['reduction_pct']}% reduction)")
        return manifest

    def _find_match(self, ph, dh, window):
        """Return match info if (ph, dh) is a near-duplicate of anything in the window."""
        for prev_ph, prev_dh, prev_id in reversed(window):
            p_dist, d_dist = ph - prev_ph, dh - prev_dh
            p_hit = p_dist <= self.cfg.phash_threshold
            d_hit = d_dist <= self.cfg.dhash_threshold
            is_dup = (p_hit and d_hit) if self.cfg.require_both else (p_hit or d_hit)
            if is_dup:
                return {"duplicate_of": prev_id,
                        "phash_distance": int(p_dist),
                        "dhash_distance": int(d_dist)}
        return None


deduper = PerceptualDedupAgent(CFG)
manifest = deduper.run(manifest)


### Tuning the thresholds

With `hash_size=16` the hashes are 256-bit, so the thresholds are out of 256:

| Threshold | Behavior |
|---|---|
| 4–8 | Very strict — only compression-noise-level differences count as duplicates |
| **10–14 (default 12)** | Slides with identical layout + text collapse; a new bullet point survives |
| 20+ | Aggressive — animated slide builds may collapse into one frame |

Re-run the cell above after changing `CFG.phash_threshold` / `CFG.dhash_threshold` — but **move the duplicates back first** with the helper below, since dedup compares against files on disk.

In [ ]:
# @title (optional) Reset dedup — move duplicates back and re-run with new thresholds
def reset_dedup():
    moved = 0
    for p in Path(CFG.duplicates_dir).glob(f"*.{CFG.image_format}"):
        shutil.move(str(p), Path(CFG.frames_dir) / p.name); moved += 1
    for p in Path(CFG.kept_dir).glob(f"*.{CFG.image_format}"):
        shutil.move(str(p), Path(CFG.frames_dir) / p.name); moved += 1
    for rec in manifest["frames"]:
        rec["status"], rec["dedup"] = "extracted", None
        rec["path"] = str(Path(CFG.frames_dir) / (rec["frame_id"] + "." + CFG.image_format))
    print(f"Moved {moved} frames back to frames_raw. "
          f"Now edit CFG thresholds and re-run the dedup cell.")

# reset_dedup()   # ← uncomment to use


In [ ]:
# @title 6. Visual audit — kept vs duplicate frames, and a timeline
import matplotlib.pyplot as plt

kept = [f for f in manifest["frames"] if f["status"] == "kept"]
dups = [f for f in manifest["frames"] if f["status"] == "duplicate"]

# --- contact sheet of kept frames (first 12) ---
sample = kept[:12]
if sample:
    cols = 4
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 3 * rows))
    for ax, rec in zip(axes.flat, sample):
        ax.imshow(Image.open(rec["path"]))
        ax.set_title(f'{rec["frame_id"]} @ {rec["timestamp_hms"]}', fontsize=9)
        ax.axis("off")
    for ax in axes.flat[len(sample):]:
        ax.axis("off")
    plt.suptitle("Kept frames (first 12)")
    plt.tight_layout(); plt.show()

# --- timeline: which 2-minute slots survived ---
fig, ax = plt.subplots(figsize=(16, 1.6))
for rec in manifest["frames"]:
    color = "#1D9E75" if rec["status"] == "kept" else "#D3D1C7"
    ax.axvspan(rec["timestamp_sec"] / 60,
               rec["timestamp_sec"] / 60 + CFG.frame_interval_sec / 60,
               color=color, alpha=0.9)
ax.set_xlim(0, VIDEO_INFO["duration_sec"] / 60)
ax.set_yticks([]); ax.set_xlabel("minutes")
ax.set_title("Timeline — green slots kept, gray slots deduped")
plt.show()

print(f"{len(kept)} kept · {len(dups)} duplicates")


In [ ]:
# @title 7. Spot-check a duplicate decision (side-by-side)
def inspect_duplicate(n: int = 0):
    """Show the n-th duplicate next to the frame it matched."""
    dup_list = [f for f in manifest["frames"] if f["status"] == "duplicate"]
    if not dup_list:
        print("No duplicates recorded."); return
    d = dup_list[min(n, len(dup_list) - 1)]
    original = next(f for f in manifest["frames"] if f["frame_id"] == d["dedup"]["duplicate_of"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, rec, tag in ((axes[0], original, "KEPT"), (axes[1], d, "DUPLICATE")):
        ax.imshow(Image.open(rec["path"]))
        ax.set_title(f'{tag}: {rec["frame_id"]} @ {rec["timestamp_hms"]}')
        ax.axis("off")
    plt.suptitle(f'pHash dist {d["dedup"]["phash_distance"]} · dHash dist {d["dedup"]["dhash_distance"]}')
    plt.show()

inspect_duplicate(0)


## Package the output for Stage 3 (Qwen content extraction)

The next agent needs exactly two things: `frames_kept/` and `manifest.json`. The cell below zips them; download the archive or copy it to Drive.

In [ ]:
# @title 8. Zip surviving frames + manifest

archive = Path(CFG.work_dir) / "stage2_output"
if archive.with_suffix(".zip").exists():
    archive.with_suffix(".zip").unlink()

staging = Path(CFG.work_dir) / "_staging"
if staging.exists(): shutil.rmtree(staging)
staging.mkdir()
shutil.copytree(CFG.kept_dir, staging / "frames_kept")
shutil.copy(CFG.manifest_path, staging / "manifest.json")
zip_path = shutil.make_archive(str(archive), "zip", staging)
shutil.rmtree(staging)

print("Created:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")

# Option 1 — browser download:
# from google.colab import files; files.download(zip_path)

# Option 2 — copy to Drive (if mounted):
# shutil.copy(zip_path, "/content/drive/MyDrive/pipeline/stage2_output.zip")


## What the manifest looks like after these stages

```json
{
  "video": {"path": "...", "duration_sec": 3720.5, "width": 1280, ...},
  "stages": {
    "extraction":        {"mode": "fast", "frame_count": 32, ...},
    "perceptual_dedup":  {"kept": 14, "duplicates": 18, "reduction_pct": 56.3, ...}
  },
  "frames": [
    {"frame_id": "frame_0001", "timestamp_sec": 0,   "timestamp_hms": "00:00:00",
     "status": "kept",      "phash": "...", "dhash": "...", "dedup": null, "path": ".../frames_kept/frame_0001.jpg"},
    {"frame_id": "frame_0002", "timestamp_sec": 120, "timestamp_hms": "00:02:00",
     "status": "duplicate", "dedup": {"duplicate_of": "frame_0001", "phash_distance": 3, "dhash_distance": 5}, ...}
  ]
}
```

**Next stage** (content extraction agent) iterates over `frames` where `status == "kept"`, calls your fine-tuned Qwen model per frame, and appends an `extracted_content` field to each record — the timestamps and dedup provenance ride along untouched.